# Certified bounds on a local expectation value

A two-site Gaussian PEPS: `L` carries `[physical | virtual]`, `Lbar` carries `[virtual | environment]`, and contracting them gives the covariance of the whole system.  The patch is the physical legs of `L`.

`GaussianCGO` spans the patch subspace with the support of the reduced state on it, $V_L=\mathrm{Im}\,\rho_L$: the state factorizes through the virtual bond, so its reduction onto the patch is a mixture of the patch states of every boundary condition and lives in their span.  The bounds are the two ends of $\min/\max\mathrm{Tr}(\sigma B_L)$ over the states of $V_L$ with $\mathrm{Tr}(\sigma[H_L,A_i])=0$, and they bracket the exact expectation because the state itself is in $V_L$.


In [ ]:
import torch

from FreeFermion.GfPEPS import Bond, Graph, GfPEPS
from FreeFermion.linalg import RandPureCov
from CGO.gaussian import GaussianCGO

torch.manual_seed(0)

PHYSICAL, VIRTUAL, ENVIRONMENT = 8, 4, 8

sub = RandPureCov(PHYSICAL + VIRTUAL)                # [physical | virtual]
environment = RandPureCov(VIRTUAL + ENVIRONMENT)     # [virtual | environment]
whole = GfPEPS.from_covariances(
    Graph(2, (Bond(0, 1, VIRTUAL),)), [PHYSICAL, ENVIRONMENT],
    [sub, environment]).contract_all()[0].tensors[0]

cgo = GaussianCGO(whole, list(range(PHYSICAL)))

print(f'whole   {tuple(whole.shape)}, the contraction of the two sites')
print(f'patch   Majoranas {cgo.internal_index}')
fock = 2 ** cgo.modes
print(f'fock    {fock} = 2**(physical/2)')
print(f'V_L     {cgo.size} of {fock}, the support of the reduced state on the patch')


## 1. The bounds across the bond dimension

Every `whole` differs in the virtual bond, so the patch block differs too.  `basic` is the eigenvalue range of $B_L$ and `CGO` adds the commutator constraints; `in` says whether the exact expectation lies inside.


In [ ]:
OBSERVABLE = [(1.0, [0, 1, 2, 3])]

print(f'{"bond":>5} {"dim V_L":>8} {"<O>":>11} {"basic":>23} {"in":>5} | CGO 1..4 (width, in)')
for bond in (2, 4, 6, 8):
    torch.manual_seed(0)
    sub = RandPureCov(PHYSICAL + bond)
    environment = RandPureCov(bond + ENVIRONMENT)
    whole = GfPEPS.from_covariances(
        Graph(2, (Bond(0, 1, bond),)), [PHYSICAL, ENVIRONMENT],
        [sub, environment]).contract_all()[0].tensors[0]
    instance = GaussianCGO(whole, list(range(PHYSICAL)))
    exact = instance.by_wick(*OBSERVABLE)
    low, high = instance.basic_CGO(*OBSERVABLE)
    cells = []
    for degree in (1, 2, 3, 4):
        lower, upper = instance.CGO(*OBSERVABLE, degree=degree, eps=1e-9)
        inside = lower - 1e-6 <= exact <= upper + 1e-6
        cells.append(f'{degree}:{upper - lower:.1e},{"Y" if inside else "N"}')
    print(f'{bond:>5} {instance.size:>8} {exact:>11.6f} [{low:+.6f},{high:+.6f}] '
          f'{str(low - 1e-9 <= exact <= high + 1e-9):>5} | ' + '  '.join(cells))


## 2. Notes

* The spanning space is the support of the reduced state on the patch, so the state lies in it by construction and the eigenvalues of $B_L$ bracket the expectation: the `in` column is true everywhere, at every bond and every degree.
* The virtual bond caps the dimension: a bond of $\chi$ Majoranas leaves at most $2^{\chi/2}$ directions, which is why `dim V_L` is 2, 4, 8 and 16 down the table.
* The tightening is weak here.  $H_L$ is the flat Hamiltonian of a random patch block, so it couples every pair of Majoranas, $[H_L,\cdot]$ spans few directions and the constraints barely cut.  A local $H$ with an interior gives the constraints something to work with.
* `SCS` at `eps=1e-9` solved every row; `CLARABEL` does not solve programs of this size.
